In [1]:
import io
import numpy as np
import tensorflow as tf
from PIL import Image
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download
import pandas as pd
from tqdm import tqdm

# Load data

In [2]:
df = pd.read_csv('/kaggle/input/datasets/beshoyshohdy/skin-data/tensorflow_original/tensorflow_original/metadata.csv')

In [3]:
df.isna().sum()

image_path                           0
diagnosis                            0
source                               0
sha256                               0
lesion_id                        14917
group_id                             0
age                                  0
sex                                  0
anatom_site_general_challenge        0
split                                0
age_missing                          0
site_missing                         0
sex_missing                          0
label                                0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
import os

df["image_path"] = df["image_path"].apply(os.path.basename)

In [6]:
df["image_path"] = '/kaggle/input/datasets/beshoyshohdy/skin-data/tensorflow_original/tensorflow_original' + '/' + df["split"] + "/" + df['diagnosis'] + '_' + df["image_path"]

In [7]:
df.head(10)

,image_path,diagnosis,source,sha256,lesion_id,group_id,age,sex,anatom_site_general_challenge,split,age_missing,site_missing,sex_missing,label
0,/kaggle/input/datasets/beshoyshohdy/skin-data/...,mel,ISIC2019,fc30b8a43cc0ae238fcaba71155d831dfb86e9eb2ccc57...,NaN,ISIC_IMAGE_ISIC_0000002.jpg,60.0,female,upper extremity,train,0,0,0,5
1,/kaggle/input/datasets/beshoyshohdy/skin-data/...,mel,ISIC2019,1fe6ec92526cf04b67cd5493606922a60faf7104a1a144...,NaN,ISIC_IMAGE_ISIC_0000004.jpg,80.0,male,posterior torso,val,0,0,0,5
2,/kaggle/input/datasets/beshoyshohdy/skin-data/...,mel,ISIC2019,b6a83b742377478ca79a2bb4923114ae78352fbb38b61c...,NaN,ISIC_IMAGE_ISIC_0000013.jpg,30.0,female,posterior torso,train,0,0,0,5
3,/kaggle/input/datasets/beshoyshohdy/skin-data/...,mel,ISIC2019,61f7243c878a24ac4a717f3f29931dc0c2d6a8cff11844...,NaN,ISIC_IMAGE_ISIC_0000022_downsampled.jpg,55.0,female,lower extremity,test,0,0,0,5
4,/kaggle/input/datasets/beshoyshohdy/skin-data/...,mel,ISIC2019,05806758b5db7bc44cba270d868c55dc2957581f5a2f0c...,NaN,ISIC_IMAGE_ISIC_0000026_downsampled.jpg,30.0,female,lower extremity,train,0,0,0,5
5,/kaggle/input/datasets/beshoyshohdy/skin-data/...,mel,ISIC2019,c3217f21981a20395f2a58ad2a531796ba3404af925d58...,NaN,ISIC_IMAGE_ISIC_0000029_downsampled.jpg,45.0,female,posterior torso,train,0,0,0,5
6,/kaggle/input/datasets/beshoyshohdy/skin-data/...,mel,ISIC2019,2ce4dd519c00f1269145a6b938c196a09fc2b0e14acd7e...,NaN,ISIC_IMAGE_ISIC_0000030_downsampled.jpg,55.0,female,lower extremity,train,0,0,0,5
7,/kaggle/input/datasets/beshoyshohdy/skin-data/...,mel,ISIC2019,46160c394f56964ceaec586f22aa9adfe15a5d947d2420...,NaN,ISIC_IMAGE_ISIC_0000031_downsampled.jpg,70.0,male,upper extremity,test,0,0,0,5
8,/kaggle/input/datasets/beshoyshohdy/skin-data/...,mel,ISIC2019,82e61ab921cd567bfc3b24d72b607bb6d5130c2208f3cc...,NaN,ISIC_IMAGE_ISIC_0000035_downsampled.jpg,25.0,female,lower extremity,test,0,0,0,5
9,/kaggle/input/datasets/beshoyshohdy/skin-data/...,mel,ISIC2019,9eae2e63894502b9a2d3f5ebcae02ae7bd9ada25da6b5a...,NaN,ISIC_IMAGE_ISIC_0000036_downsampled.jpg,70.0,male,upper extremity,val,0,0,0,5


In [8]:
model_df = df[
    [
        'image_path',
        'diagnosis',
        'age',
        'sex',
        'anatom_site_general_challenge',
        'split',
        'label'
    ]
].copy()

In [9]:
model_df['image_path']

0        /kaggle/input/datasets/beshoyshohdy/skin-data/...
1        /kaggle/input/datasets/beshoyshohdy/skin-data/...
2        /kaggle/input/datasets/beshoyshohdy/skin-data/...
3        /kaggle/input/datasets/beshoyshohdy/skin-data/...
4        /kaggle/input/datasets/beshoyshohdy/skin-data/...
                               ...                        
24925    /kaggle/input/datasets/beshoyshohdy/skin-data/...
24926    /kaggle/input/datasets/beshoyshohdy/skin-data/...
24927    /kaggle/input/datasets/beshoyshohdy/skin-data/...
24928    /kaggle/input/datasets/beshoyshohdy/skin-data/...
24929    /kaggle/input/datasets/beshoyshohdy/skin-data/...
Name: image_path, Length: 24930, dtype: object

In [10]:
train = model_df[model_df['split'] == 'train']
val = model_df[model_df['split'] == 'val']
test = model_df[model_df['split'] == 'test']

print(f"train: {train.shape}")
print(f"val: {val.shape}")
print(f"test: {test.shape}")

train: (17464, 7)
val: (3713, 7)
test: (3753, 7)


# balance data

In [15]:
train['diagnosis'].value_counts()

diagnosis
nv       6444
mel      4004
bcc      2612
bkl      2607
ak        600
scc       448
vasc      281
df        238
akiec     230
Name: count, dtype: int64

In [21]:
train_balanced = (
    train.groupby("diagnosis", group_keys=False)
      .apply(lambda g: g.sample(n=min(len(g), 1500), random_state=42))
      .reset_index(drop=True)
)

print(train_balanced["diagnosis"].value_counts())

diagnosis
bcc      1500
nv       1500
bkl      1500
mel      1500
ak        600
scc       448
vasc      281
df        238
akiec     230
Name: count, dtype: int64


/tmp/ipykernel_174/1353385173.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), 1500), random_state=42))


In [22]:
train_balanced

,image_path,diagnosis,age,sex,anatom_site_general_challenge,split,label
0,/kaggle/input/datasets/beshoyshohdy/skin-data/...,ak,50.0,female,lower extremity,train,0
1,/kaggle/input/datasets/beshoyshohdy/skin-data/...,ak,70.0,male,head/neck,train,0
2,/kaggle/input/datasets/beshoyshohdy/skin-data/...,ak,60.0,male,anterior torso,train,0
3,/kaggle/input/datasets/beshoyshohdy/skin-data/...,ak,70.0,male,head/neck,train,0
4,/kaggle/input/datasets/beshoyshohdy/skin-data/...,ak,60.0,male,head/neck,train,0
...,...,...,...,...,...,...,...
7792,/kaggle/input/datasets/beshoyshohdy/skin-data/...,vasc,35.0,male,trunk,train,8
7793,/kaggle/input/datasets/beshoyshohdy/skin-data/...,vasc,35.0,female,lower extremity,train,8
7794,/kaggle/input/datasets/beshoyshohdy/skin-data/...,vasc,70.0,female,lower extremity,train,8
7795,/kaggle/input/datasets/beshoyshohdy/skin-data/...,vasc,60.0,male,trunk,train,8


In [33]:
train_balanced['diagnosis'].value_counts()

diagnosis
bcc      1500
nv       1500
bkl      1500
mel      1500
ak        600
scc       448
vasc      281
df        238
akiec     230
Name: count, dtype: int64

# Google embedding model

In [11]:
tf.config.set_visible_devices([], "GPU")   # hide the GPU from TensorFlow

login(UserSecretsClient().get_secret('hf')) # secret key hf

path = snapshot_download("google/derm-foundation")

model = tf.saved_model.load(path)
infer = model.signatures["serving_default"]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

In [12]:
def image_to_example(img_path):
    img = Image.open(img_path).convert("RGB")   # handles RGBA, grayscale, etc.
    buf = io.BytesIO()
    img.save(buf, format="PNG")                 # model expects PNG bytes
    png_bytes = buf.getvalue()

    example = tf.train.Example()
    feats = example.features.feature
    feats["image/encoded"].bytes_list.value.append(png_bytes)
    feats["image/format"].bytes_list.value.append(b"PNG")
    return example.SerializeToString()

infer = model.signatures["serving_default"]

In [13]:
def embed_paths(paths, batch_size=16):
    embs = []
    for i in tqdm(range(0, len(paths), batch_size)):
        batch = [image_to_example(p) for p in paths[i:i+batch_size]]
        out = infer(inputs=tf.constant(batch))
        embs.append(out["embedding"].numpy())
    return np.concatenate(embs)   # shape (N, 6144)  

In [23]:
train_embeddings = embed_paths(
    train_balanced["image_path"],
    batch_size=16
)

np.save("/kaggle/working/train_embeddings.npy", train_embeddings)

100%|██████████| 488/488 [6:53:05<00:00, 50.79s/it]  


In [28]:
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras import layers, regularizers, callbacks, Model

# X: Derm Foundation embeddings, shape (N, 6144)
# y: disease labels, e.g. "nv", "mel", "bcc", etc.

X = np.asarray( train_embeddings, dtype=np.float32)
y = np.asarray(train_balanced['label'])

print("Embeddings:", X.shape)
print("Labels:", y.shape)


# Split into train (70%), validation (15%), test (15%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

# Standardize using training data only
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Embeddings: (7797, 6144)
Labels: (7797,)
Train: (5457, 6144)
Validation: (1170, 6144)
Test: (1170, 6144)


In [52]:
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model

strategy = tf.distribute.MirroredStrategy()

print("Number of GPUs:", strategy.num_replicas_in_sync)

strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    inputs = layers.Input(shape=(X_train.shape[1],))

    x = layers.Dense(
        256,
        kernel_regularizer=regularizers.l2(1e-4)
    )(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.40)(x)

    x = layers.Dense(
        128,
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Dense(64, activation="relu")(x)

    outputs = layers.Dense(
        9,
        activation="softmax"
    )(x)

    model = Model(inputs, outputs, name="derm_foundation_ann")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=3e-4
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

model.summary()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)
Number of GPUs: 1
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


Model: "derm_foundation_ann"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 6144)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 256)            │     1,573,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_15 (Activation)      │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_16 (Activation)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 9)              │           585 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,616,393 (6.17 MB)

 Trainable params: 1,615,625 (6.16 MB)

 Non-trainable params: 768 (3.00 KB)

In [53]:
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

print("Class weights:", class_weights)

Class weights: {np.int64(0): np.float64(1.4436507936507936), np.int64(1): np.float64(3.7660455486542443), np.int64(2): np.float64(0.5774603174603175), np.int64(3): np.float64(0.5774603174603175), np.int64(4): np.float64(3.6526104417670684), np.int64(5): np.float64(0.5774603174603175), np.int64(6): np.float64(0.5774603174603175), np.int64(7): np.float64(1.93716719914803), np.int64(8): np.float64(3.077834179357022)}


In [54]:
early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=9,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-8,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

Epoch 1/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.3451 - loss: 1.8019 - val_accuracy: 0.5009 - val_loss: 1.5241 - learning_rate: 3.0000e-04
Epoch 2/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5153 - loss: 1.3953 - val_accuracy: 0.5342 - val_loss: 1.4024 - learning_rate: 3.0000e-04
Epoch 3/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5520 - loss: 1.2572 - val_accuracy: 0.5547 - val_loss: 1.3404 - learning_rate: 3.0000e-04
Epoch 4/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5793 - loss: 1.1656 - val_accuracy: 0.5735 - val_loss: 1.2873 - learning_rate: 3.0000e-04
Epoch 5/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6029 - loss: 1.0923 - val_accuracy: 0.5932 - val_loss: 1.2305 - learning_rate: 3.0000e-04
Epoch 6/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6264 - loss: 1.0151 - val_accuracy: 0.5974 - val_loss: 1.2098 - learning_rate: 3.0000e-04
Epoch 7/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 

In [55]:
test_loss, test_accuracy = model.evaluate(
    X_test, y_test, verbose=0
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

y_prob = model.predict(X_test, batch_size=32)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(9),
    zero_division=0
))

Test loss: 0.9392
Test accuracy: 0.7214
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
              precision    recall  f1-score   support

           0       0.50      0.66      0.57        90
           1       0.40      0.71      0.51        35
           2       0.80      0.80      0.80       225
           3       0.78      0.67      0.72       225
           4       0.68      0.83      0.75        36
           5       0.73      0.68      0.70       225
           6       0.81      0.79      0.80       225
           7       0.55      0.45      0.49        67
           8       0.87      0.95      0.91        42

    accuracy                           0.72      1170
   macro avg       0.68      0.73      0.69      1170
weighted avg       0.73      0.72      0.72      1170

